In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_csv("PS_train.csv")

df.head()

,content,labels
0,தென்காசி தொகுதி புதிய தமிழகம் கட்சி வேட்பாளர் ...,Neutral
1,அண்ணன் இதனை சூசகமாக 11 மாதங்கள் முன்பே பேட்டிய...,Substantiated
2,ஒரு வருடம் ஆகி விட்டது இந்த துயரம் நேர்ந்து......,Opinionated
3,"எடப்பாடியை கண்டுகொள்ளாத ""எடப்பாடி""🫢\n ---\nஆதர...",Positive
4,எங்களின் அரசியல் அடுத்த தலைமுறைக்குமானது \n#மக...,Opinionated


In [ ]:
valid_labels = ["Negative", "Neutral", "Positive"]

df = df[df["labels"].isin(valid_labels)].reset_index(drop=True)

print(df["labels"].value_counts())

label_mapping = {
    "Positive": "Positive",
    "Neutral": "Neutral",
    "Negative": "Negative",
    "Opinionated": "Neutral",
    "Substantiated": "Neutral",
    "Sarcastic": "Negative",
    "None of the above": "Neutral"
}

df["sentiment_label"] = df["labels"].map(label_mapping)
df.head()


labels
Neutral     637
Positive    575
Negative    406
Name: count, dtype: int64


In [ ]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME
)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [ ]:
y_true = []
y_pred = []

for text, label in zip(df["content"], df["labels"]):
    result = sentiment_pipeline(
        text,
        truncation=True,
        max_length=512
    )[0]

    y_pred.append(result["label"].capitalize())
    y_true.append(label)

In [ ]:
print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

Accuracy: 0.39184177997527814

Classification Report:

              precision    recall  f1-score   support

    Negative       0.12      0.00      0.00       406
     Neutral       0.39      0.97      0.56       637
    Positive       0.42      0.03      0.06       575

    accuracy                           0.39      1618
   macro avg       0.31      0.33      0.21      1618
weighted avg       0.34      0.39      0.24      1618



In [ ]:
def predict_example(text):
    result = sentiment_pipeline(
        text,
        truncation=True,
        max_length=512
    )[0]

    print("Text:", text)
    print("Prediction:", result["label"].capitalize())
    print("Confidence:", round(result["score"], 4))

In [ ]:
predict_example("இன்று சட்டமன்ற கூட்டம் நடைபெற்றது")

Text: இன்று சட்டமன்ற கூட்டம் நடைபெற்றது
Prediction: Neutral
Confidence: 0.6907


In [ ]:
predict_example("இந்த அரசு நல்ல திட்டங்களை செயல்படுத்தியுள்ளது")

Text: இந்த அரசு நல்ல திட்டங்களை செயல்படுத்தியுள்ளது
Prediction: Neutral
Confidence: 0.7087


In [ ]:
predict_example("இந்த அரசு மிகவும் மோசமாக செயல்படுகிறது")

Text: இந்த அரசு மிகவும் மோசமாக செயல்படுகிறது
Prediction: Neutral
Confidence: 0.6955


In [ ]:
predict_example("""
இந்த தேர்தலில் மக்கள் இந்த அரசின் செயல்பாடுகளை
கடுமையாக விமர்சித்து வருகின்றனர்
""")

Text: 
இந்த தேர்தலில் மக்கள் இந்த அரசின் செயல்பாடுகளை
கடுமையாக விமர்சித்து வருகின்றனர்

Prediction: Neutral
Confidence: 0.6972


In [ ]:
predict_example(
    "Worst government ever. Completely failed and people are suffering badly."
)

Text: Worst government ever. Completely failed and people are suffering badly.
Prediction: Negative
Confidence: 0.9566


In [ ]:
predict_example(
    "This is the best government ever. Amazing performance and excellent leadership."
)

Text: This is the best government ever. Amazing performance and excellent leadership.
Prediction: Positive
Confidence: 0.9856


In [ ]:
predict_example("இந்த அரசு மிக மோசமானது. இது முழுமையாக தோல்வியடைந்துள்ளது.")

Text: இந்த அரசு மிக மோசமானது. இது முழுமையாக தோல்வியடைந்துள்ளது.
Prediction: Neutral
Confidence: 0.6613


In [ ]:
predict_example("இந்த அரசு சிறப்பாக செயல்பட்டு மக்களுக்கு நல்ல சேவையை வழங்கியுள்ளது.")

Text: இந்த அரசு சிறப்பாக செயல்பட்டு மக்களுக்கு நல்ல சேவையை வழங்கியுள்ளது.
Prediction: Neutral
Confidence: 0.6904
